# 04 - HuggingFace Datasets 完整教學（2026 版）

## 學習目標

1. 使用 `load_dataset` 從 HuggingFace Hub 載入公開資料集
2. 掌握資料集的基本操作：查看、切分、過濾、選取
3. 理解 `map(batched=True)` 的運作原理，學會高效 tokenize
4. 使用 `DataCollatorWithPadding` 實現動態 padding，取代固定長度截斷
5. 掌握本地資料（CSV / JSON / pandas DataFrame）轉換為 Dataset 的方法
6. 學會將處理後的資料集存入磁碟並重新載入

## 前置知識

- 已完成 `01-Getting Started` 與 `03-Tokenizer` 的教學
- 了解 HuggingFace `AutoTokenizer` 的基本使用

## 與相鄰 Notebook 的銜接

- 上一章：`../03-Tokenizer/03 Tokenizer.ipynb` — Tokenizer 原理與編碼方式
- 下一章：`../05-Trainer/05 Trainer.ipynb` — 以 Trainer 進行監督學習訓練

本章產出的 `tokenized_dataset` 與 `DataCollatorWithPadding` 是下一章 Trainer 訓練流程的直接輸入。

## 0. 環境安裝與版本鎖定

以下為 2026 年全 repo 統一的最低版本需求。固定版本下界可確保 `batched=True` 的 Arrow 多執行緒加速、`stratify_by_column` 等 API 可用。

In [ ]:
# Install pinned dependencies
# Run once; restart kernel if prompted
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "evaluate>=0.4" \
    "torch>=2.4"

In [ ]:
# Verify installed versions
import transformers, datasets, accelerate, safetensors, evaluate, torch

for pkg in [transformers, datasets, accelerate, safetensors, evaluate, torch]:
    print(f"{pkg.__name__}: {pkg.__version__}")

## 1. datasets 基本使用

`datasets` 是 HuggingFace 提供的資料集管理函式庫，底層以 Apache Arrow 格式儲存資料，具備：

- **零拷貝讀取**：Arrow 記憶體映射，切片不複製資料
- **跨平台快取**：下載後自動快取於 `~/.cache/huggingface/datasets`
- **原生批次處理**：`map(batched=True)` 可比逐筆處理快 3–5 倍

### 1.1 從 HuggingFace Hub 下載資料集

In [ ]:
from datasets import load_dataset

# Load a Chinese news headline dataset from the Hub
# Returns a DatasetDict with train/validation/test splits
datasets = load_dataset("madao33/new-title-chinese")
datasets

### 1.2 下載資料集包中的特定子任務

某些 Hub 資料集（如 `super_glue`、`glue`）包含多個子任務（config）。傳入第二個字串參數即可指定子任務名稱。

In [ ]:
# BoolQ is a yes/no question-answering task inside the SuperGLUE benchmark
boolq_dataset = load_dataset("super_glue", "boolq")
boolq_dataset

### 1.3 只下載特定 split 或切片

`split` 參數支援豐富的切片語法，與 Python list 切片類似，可節省下載時間與記憶體。

In [ ]:
# Only the training split
dataset_train = load_dataset("madao33/new-title-chinese", split="train")
print("full train:", len(dataset_train))

# Rows 10 to 99 (exclusive upper bound)
dataset_slice = load_dataset("madao33/new-title-chinese", split="train[10:100]")
print("rows 10-99:", len(dataset_slice))

# First 50% of the training split
dataset_half = load_dataset("madao33/new-title-chinese", split="train[:50%]")
print("first 50%:", len(dataset_half))

# Load two non-overlapping halves as a list of Dataset
two_halves = load_dataset("madao33/new-title-chinese", split=["train[:50%]", "train[50%:]"])
print("two halves lengths:", [len(d) for d in two_halves])

## 2. 查看資料集內容

Dataset 的欄位存取方式類似 pandas DataFrame，但底層是 Arrow，效率更高。

In [ ]:
# Re-load full DatasetDict for the rest of the notebook
datasets = load_dataset("madao33/new-title-chinese")

# Keys of a single row
print("columns:", datasets["train"].column_names)

# Feature schema (dtype, sequence info)
print("\nfeatures:")
print(datasets["train"].features)

In [ ]:
# First row as dict
datasets["train"][0]

In [ ]:
# Slice rows 0-3: returns dict of lists (column-oriented)
datasets["train"][:4]

In [ ]:
# Access a single column for the first 5 rows
datasets["train"]["title"][:5]

## 3. 資料集切分

### 3.1 隨機切分

`train_test_split` 回傳一個新的 `DatasetDict`，含 `train` 與 `test` 兩個 split。

**2026 慣例**：永遠傳入 `seed` 確保可重現性。

In [ ]:
# Random split with fixed seed for reproducibility
dataset = datasets["train"]
split_result = dataset.train_test_split(test_size=0.1, seed=42)
split_result

### 3.2 分層切分（stratified split）

對分類任務而言，隨機切分可能導致某些類別在 test 集中比例失真。`stratify_by_column` 確保每個類別在 train/test 中的比例相同。

**WHY**：若訓練集正負比例為 9:1 但測試集為 7:3，評測結果將失真，影響模型選擇。

In [ ]:
# BoolQ has a binary label; stratify ensures balanced class ratio in both splits
boolq_train = boolq_dataset["train"]
dataset_train_valid = boolq_train.train_test_split(
    test_size=0.1,
    stratify_by_column="label",  # preserve class ratio
    seed=42
)

# Rename 'test' to 'valid' to reflect its actual role
dataset_train_valid["valid"] = dataset_train_valid.pop("test")
dataset_train_valid

## 4. 資料挑選與過濾

### 4.1 `select` — 按索引挑選

In [ ]:
# Select specific rows by index (returns a new Dataset, no copy of data)
selected = datasets["train"].select([0, 20])
print(selected[0])
print(selected[1])

### 4.2 `filter` — 按條件過濾

`filter` 接受一個回傳布林值的函式，逐筆（或批次）篩選資料。

In [ ]:
# Keep only rows where the title contains the keyword '中国'
filter_dataset = datasets["train"].filter(
    lambda example: "中国" in example["title"]
)
print(f"Filtered rows: {len(filter_dataset)}")
filter_dataset["title"][:10]

## 5. 資料 Mapping（map）

`map` 是 datasets 中最核心的轉換函式，用途等同 pandas 的 `apply`，但具備：

- **Arrow 批次加速**：`batched=True` 讓函式一次處理整批資料，省去逐筆呼叫的 Python overhead，速度通常快 3–5 倍
- **多核心平行**：`num_proc=N` 啟動多個 worker process，進一步加速
- **惰性快取**：相同函式 + 相同資料的結果會自動快取

### 5.1 簡單欄位轉換

In [ ]:
def add_prefix(example):
    # Prepend a fixed string to the title column
    example["title"] = "Prefix: " + example["title"]
    return example

prefix_dataset = datasets.map(add_prefix)
prefix_dataset["train"][:3]["title"]

### 5.2 Tokenizer 預處理：逐筆 vs 批次 vs 多核心

以下三種寫法在功能上等價，但效能差距顯著。

| 寫法 | 說明 | 速度 |
|---|---|---|
| `map(fn)` | 逐筆（row-by-row），最慢 | 1x |
| `map(fn, num_proc=N)` | 多 process 但仍逐筆 | ~Nx |
| `map(fn, batched=True)` | 批次 Arrow，最快 | 3–5x |
| `map(fn, batched=True, num_proc=N)` | 批次 + 多核心 | 最快 |

**為什麼 `batched=True` 最快**：Tokenizer 的 `__call__` 本身已對批次輸入做 Rust 層加速；Arrow 格式讓批次切割無需複製記憶體，純 Python 的逐筆呼叫開銷完全消失。

In [ ]:
import os
from transformers import AutoTokenizer

model_id = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_id)

num_cpus = os.cpu_count()
print(f"Available CPUs: {num_cpus}")

In [ ]:
def preprocess_function(examples):
    """
    Tokenize both content and title.
    When batched=True, `examples` is a dict of lists;
    tokenizer handles the list natively via its Rust backend.
    """
    model_inputs = tokenizer(
        examples["content"],
        max_length=512,
        truncation=True
        # NOTE: no padding here — DataCollatorWithPadding will pad dynamically per batch
    )
    labels = tokenizer(
        examples["title"],
        max_length=32,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# Baseline: row-by-row (slow)
%%time
processed_row = datasets.map(preprocess_function)
print(processed_row)

In [ ]:
# Multi-process, still row-by-row
%%time
processed_multiproc = datasets.map(preprocess_function, num_proc=2)
print(processed_multiproc)

In [ ]:
# batched=True: Arrow batch processing, 3-5x faster
%%time
processed_batched = datasets.map(preprocess_function, batched=True)
print(processed_batched)

In [ ]:
# Best practice: batched + multi-proc + drop original columns
%%time
processed_datasets = datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=datasets["train"].column_names,  # drop title & content columns
    num_proc=min(4, num_cpus),                      # cap at 4 to avoid memory pressure
)
processed_datasets

## 6. 資料集儲存與載入

處理後的 Dataset 以 Arrow 格式存入磁碟，重新載入時**不需要重新 tokenize**，適合在訓練前預先處理好整個資料集。

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("./processed_data")

# Save to disk (Arrow format, no serialization overhead)
processed_datasets.save_to_disk(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR.resolve()}")

In [ ]:
from datasets import load_from_disk

# Reload without re-tokenizing
reloaded = load_from_disk(OUTPUT_DIR)
print(reloaded)
print("\nSample row:", reloaded["train"][0])

## 7. 載入本地端資料集

**2026 慣例**：本地資料路徑依以下優先順序處理：

1. HuggingFace Hub model/dataset ID（優先）
2. 相對路徑 + `pathlib.Path`（本地開發）
3. 環境變數 `DATA_DIR`（CI / 生產環境）

以下示範以 `DATA_DIR` 環境變數控制本地資料路徑。

### 7.1 讀取 CSV 檔案

In [ ]:
import os
from pathlib import Path

# Set DATA_DIR to your local data directory, or use a default relative path
DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))

# Example: load a CSV sentiment corpus
# Expected file: $DATA_DIR/ChnSentiCorp_htl_all.csv
csv_path = DATA_DIR / "ChnSentiCorp_htl_all.csv"

if csv_path.exists():
    dataset_csv = load_dataset("csv", data_files=str(csv_path), split="train")
    print(dataset_csv)
else:
    print(f"[SKIP] CSV not found at {csv_path}")
    print("To run this section: set DATA_DIR env var or place the file at ./data/")

### 7.2 讀取整個資料夾中的多個 CSV 檔案

`data_dir` 參數會自動掃描指定目錄下的所有符合格式的檔案，合併為一個 Dataset。

In [ ]:
all_data_dir = DATA_DIR / "all_data"

if all_data_dir.exists():
    dataset_dir = load_dataset("csv", data_dir=str(all_data_dir), split="train")
    print(dataset_dir)
else:
    print(f"[SKIP] Directory not found at {all_data_dir}")

### 7.3 從 pandas DataFrame 建立 Dataset

當資料已透過 pandas 預處理（清洗、合併等），可直接轉換為 Dataset，保留 Arrow 加速能力。

In [ ]:
import pandas as pd
from datasets import Dataset

csv_path = DATA_DIR / "ChnSentiCorp_htl_all.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(df.head())

    # Convert directly — column dtypes are inferred automatically
    dataset_from_df = Dataset.from_pandas(df)
    print(dataset_from_df)
else:
    # Demonstrate with a synthetic DataFrame
    df = pd.DataFrame({"text": ["很好", "很差", "普通"], "label": [1, 0, 0]})
    dataset_from_df = Dataset.from_pandas(df)
    print("[Demo with synthetic data]")
    print(dataset_from_df)

### 7.4 從 Python list-of-dict 建立 Dataset

In [ ]:
# Useful for quick prototyping or unit tests
data_list = [
    {"text": "這部電影太精彩了", "label": 1},
    {"text": "服務態度很差", "label": 0},
    {"text": "還算可以", "label": 0},
]
dataset_from_list = Dataset.from_list(data_list)
print(dataset_from_list)
print(dataset_from_list[0])

### 7.5 從 JSON 檔案建立 Dataset

`field` 參數指定 JSON 中存放記錄列表的頂層鍵（若 JSON 頂層就是列表則無需 `field`）。

In [ ]:
json_path = DATA_DIR / "cmrc2018_trial.json"

if json_path.exists():
    # field='data' tells datasets where to find the list of records inside the JSON
    dataset_json = load_dataset(
        "json",
        data_files=str(json_path),
        field="data"
    )
    print(dataset_json)

    # Chain with train_test_split immediately
    split_json = dataset_json["train"].train_test_split(test_size=0.1, seed=42)
    print(split_json)
else:
    print(f"[SKIP] JSON not found at {json_path}")

## 8. DataCollatorWithPadding — 動態 Padding

### 為什麼用動態 padding？

在 `map` 中只做 `truncation=True`，**不做 padding**；再由 `DataCollatorWithPadding` 在每個 batch 內動態補到「該 batch 最長序列」的長度。

**好處**：短句子不需要被補滿整個 `max_length`，平均節省 30–50% 計算量（視資料長度分布而定）。

| 方式 | padding 長度 | 計算量 |
|---|---|---|
| 固定 `max_length=128` | 永遠 128 | 最高 |
| `DataCollatorWithPadding`（動態）| 每 batch 的最長序列 | 最低 |

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset

model_id = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load sentiment corpus; handle None reviews
if csv_path.exists():
    senti_dataset = load_dataset("csv", data_files=str(csv_path), split="train")
else:
    # Synthetic fallback for demo
    senti_dataset = Dataset.from_list([
        {"review": "環境很好，服務一流", "label": 1},
        {"review": "太差了，不會再來", "label": 0},
        {"review": "性價比還行", "label": 0},
        {"review": "房間乾淨整潔", "label": 1},
    ])

# Drop rows with null review
senti_dataset = senti_dataset.filter(lambda x: x["review"] is not None)
print(senti_dataset)

In [ ]:
def process_function(examples):
    """
    Tokenize review text.
    Key: NO padding here — DataCollatorWithPadding handles it per-batch.
    This avoids wasting compute on padding tokens that don't carry information.
    """
    tokenized = tokenizer(
        examples["review"],
        max_length=128,
        truncation=True
        # padding=False  (default, explicitly stated for clarity)
    )
    tokenized["labels"] = examples["label"]
    return tokenized

tokenized_dataset = senti_dataset.map(
    process_function,
    batched=True,
    remove_columns=senti_dataset.column_names,
)
print(tokenized_dataset)
print("\nSample (variable length, no padding yet):")
print("input_ids length:", len(tokenized_dataset[0]["input_ids"]))

In [ ]:
# Inspect the tokenizer — DataCollatorWithPadding reads pad_token_id from it
print("Tokenizer pad token:", tokenizer.pad_token)
print("Tokenizer pad token id:", tokenizer.pad_token_id)

In [ ]:
# DataCollatorWithPadding dynamically pads each batch to its longest sequence
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from torch.utils.data import DataLoader

# collate_fn=collator applies dynamic padding when forming each batch
dl = DataLoader(
    tokenized_dataset,
    batch_size=4,
    collate_fn=collator,
    shuffle=True,
)

# Inspect a few batches — each batch may have a different sequence length
for i, batch in enumerate(dl):
    print(f"Batch {i}: input_ids shape = {batch['input_ids'].shape}")
    if i >= 3:
        break

### 觀察重點

每個 batch 的 `input_ids` 第二維（序列長度）不同，因為 `DataCollatorWithPadding` 只補到「該 batch 中最長序列」的長度。這是與固定 `padding='max_length'` 最大的差異。

在 Trainer 中使用時，只需：

```python
Trainer(
    ...,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
```

Trainer 會自動處理每個 batch 的動態 padding，無需手動建立 DataLoader。

## 9. 完整流程小結

本章涵蓋了 `datasets` 從載入到模型輸入的完整流程：

```
load_dataset()              # 從 Hub 或本地載入
  └─ filter()               # 移除空值或不符條件的資料
  └─ train_test_split()     # 切分（stratify_by_column 確保類別平衡）
  └─ map(fn, batched=True)  # 高效批次 tokenize，不做 padding
  └─ save_to_disk()         # 儲存處理結果避免重複計算

DataCollatorWithPadding     # 在 DataLoader/Trainer 中動態 padding
```

## 練習題

1. 使用 `load_dataset("yelp_review_full")` 載入英文評論資料集，試著用 `stratify_by_column="label"` 切出 10% 的驗證集，並確認每個類別比例是否和原始資料集相同。

2. 比較三種 map 方式（row-by-row、`num_proc=4`、`batched=True`）在同一筆資料集上的執行時間，觀察加速比是否符合預期。

3. 修改 `process_function`，加入 `padding='max_length', max_length=128`，並與 `DataCollatorWithPadding` 的動態版本比較每個 batch 的 tensor shape，確認動態 padding 的效果。

4. 嘗試從一個 JSON 檔案載入資料（可用任何本地 JSON），用 `load_dataset("json", ...)` 載入後，立即鏈式呼叫 `.train_test_split(test_size=0.2, seed=42)`，觀察回傳的 DatasetDict 結構。